# YOLO26n baseline + GNN post-processor

This experiment keeps the trained YOLO26n baseline fixed and trains a small graph neural network to re-score YOLO candidate boxes using neighboring detections as context. It does **not** modify or retrain YOLO. A candidate is a graph node; nearby box centers are its graph neighbors.

The GNN is trained on a seeded 70% subset of the validation split. The remaining 30% selects one confidence threshold for each method. The test split is reserved for final evaluation and is not used for GNN training or threshold selection. Metrics use class-aware, one-to-one box matching at IoU 0.50; AP is computed from ranked scores at IoU 0.50 and 0.50:0.95.

Important limitation: the GNN only re-scores boxes proposed by YOLO, so it cannot recover an object for which YOLO produced no candidate box.

In [1]:
from pathlib import Path
import random
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
if not (ROOT / "datasets" / "baseline2.0").is_dir() and (ROOT.parent / "datasets" / "baseline2.0").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "datasets" / "baseline2.0"
MODEL_PATH = ROOT / "runs" / "detect" / "baseline2.0" / "weights" / "best.pt"
SPLITS = {
    "valid": (DATA_DIR / "valid" / "images", DATA_DIR / "valid" / "labels"),
    "test": (DATA_DIR / "test" / "images", DATA_DIR / "test" / "labels"),
}
CLASS_NAMES = ["3", "5", "6"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 640
MAX_DET = 1000
K_NEIGHBORS = 8
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Trained YOLO baseline not found: {MODEL_PATH}")
for split, (image_dir, label_dir) in SPLITS.items():
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"Missing {split} images or labels under {DATA_DIR}")

model = YOLO(str(MODEL_PATH))
print(f"YOLO checkpoint: {MODEL_PATH}")
print(f"Device: {DEVICE}; classes: {CLASS_NAMES}")
for split, (image_dir, _) in SPLITS.items():
    image_count = sum(p.suffix.lower() in {".jpg", ".jpeg", ".png"} for p in image_dir.iterdir())
    print(f"{split}: {image_count} images")

In [2]:
def list_images(image_dir):
    paths = sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if not paths:
        raise FileNotFoundError(f"No images found in {image_dir}")
    return paths

def read_ground_truth(image_path, label_dir):
    label_path = label_dir / f"{image_path.stem}.txt"
    if not label_path.is_file():
        raise FileNotFoundError(f"Missing label file: {label_path}")
    with Image.open(image_path) as image:
        width, height = image.size
    objects = []
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != 5:
            raise ValueError(f"Expected 5 YOLO fields at {label_path}:{line_number}")
        class_id = int(fields[0])
        if not 0 <= class_id < NUM_CLASSES:
            raise ValueError(f"Unexpected class {class_id} at {label_path}:{line_number}")
        xc, yc, bw, bh = map(float, fields[1:])
        objects.append((class_id, np.array([
            (xc - bw / 2) * width, (yc - bh / 2) * height,
            (xc + bw / 2) * width, (yc + bh / 2) * height,
        ], dtype=np.float32)))
    return objects

def predict_images(image_paths, batch_size=16):
    all_detections = []
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        results = model.predict(
            source=[str(p) for p in batch_paths],
            imgsz=IMAGE_SIZE,
            conf=0.001,
            max_det=MAX_DET,
            batch=batch_size,
            device=DEVICE,
            verbose=False,
        )
        if len(results) != len(batch_paths):
            raise RuntimeError("YOLO returned a different number of results than input images")
        for result in results:
            if result.boxes is None:
                all_detections.append({"boxes": np.empty((0, 4), dtype=np.float32), "scores": np.empty(0, dtype=np.float32), "classes": np.empty(0, dtype=np.int64)})
            else:
                all_detections.append({
                    "boxes": result.boxes.xyxy.cpu().numpy().astype(np.float32),
                    "scores": result.boxes.conf.cpu().numpy().astype(np.float32),
                    "classes": result.boxes.cls.cpu().numpy().astype(np.int64),
                })
    return all_detections

def box_iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return np.empty(0, dtype=np.float32)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = max(0.0, float(box[2] - box[0])) * max(0.0, float(box[3] - box[1]))
    area_b = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
    return intersection / np.maximum(area_a + area_b - intersection, 1e-12)

def candidate_labels(detections, ground_truth, iou_threshold=0.5):
    labels = np.zeros(len(detections["scores"]), dtype=np.float32)
    matched = set()
    order = np.argsort(-detections["scores"])
    gt_by_class = {
        c: [(i, box) for i, (gt_class, box) in enumerate(ground_truth) if gt_class == c]
        for c in range(NUM_CLASSES)
    }
    for det_index in order:
        class_id = int(detections["classes"][det_index])
        candidates = [(i, box) for i, box in gt_by_class[class_id] if i not in matched]
        if not candidates:
            continue
        gt_indices = [i for i, _ in candidates]
        gt_boxes = np.stack([box for _, box in candidates])
        ious = box_iou_one_to_many(detections["boxes"][det_index], gt_boxes)
        best = int(np.argmax(ious))
        if ious[best] >= iou_threshold:
            labels[det_index] = 1.0
            matched.add(gt_indices[best])
    return labels

def make_graph(detections, width, height, labels=None):
    boxes = detections["boxes"]
    classes = detections["classes"]
    scores = detections["scores"]
    n = len(scores)
    if n == 0:
        x = torch.empty((0, 5 + NUM_CLASSES), dtype=torch.float32)
        y = torch.empty((0,), dtype=torch.float32) if labels is not None else None
        edges = torch.empty((2, 0), dtype=torch.long)
        return {"x": x, "y": y, "edges": edges}
    scale = np.array([width, height, width, height], dtype=np.float32)
    normalized_boxes = boxes / scale
    one_hot = np.eye(NUM_CLASSES, dtype=np.float32)[classes]
    features = np.concatenate([normalized_boxes, scores[:, None], one_hot], axis=1)
    x = torch.from_numpy(features)
    centers = (normalized_boxes[:, :2] + normalized_boxes[:, 2:]) / 2
    if n <= 1:
        edges = torch.empty((2, 0), dtype=torch.long)
    else:
        distances = torch.cdist(torch.from_numpy(centers), torch.from_numpy(centers))
        distances.fill_diagonal_(float("inf"))
        k = min(K_NEIGHBORS, n - 1)
        neighbors = torch.topk(distances, k=k, largest=False).indices
        targets = torch.arange(n).repeat_interleave(k)
        sources = neighbors.reshape(-1)
        edges = torch.stack([sources, targets])
    y = torch.from_numpy(labels) if labels is not None else None
    return {"x": x, "y": y, "edges": edges}

def graphs_for_paths(paths, detections, label_dir=None):
    graphs = []
    for index, image_path in enumerate(paths):
        with Image.open(image_path) as image:
            width, height = image.size
        labels = None
        if label_dir is not None:
            labels = candidate_labels(detections[index], read_ground_truth(image_path, label_dir))
        graphs.append(make_graph(detections[index], width, height, labels))
    return graphs

valid_dir, valid_label_dir = SPLITS["valid"]
valid_paths = list_images(valid_dir)
valid_detections = predict_images(valid_paths)
valid_ground_truth = [read_ground_truth(p, valid_label_dir) for p in valid_paths]
permutation = np.random.default_rng(SEED).permutation(len(valid_paths))
fit_count = int(0.70 * len(valid_paths))
fit_indices = permutation[:fit_count].tolist()
calibration_indices = permutation[fit_count:].tolist()
valid_graphs = graphs_for_paths(valid_paths, valid_detections, label_dir=valid_label_dir)
fit_graphs = [valid_graphs[i] for i in fit_indices if len(valid_graphs[i]["y"]) > 0]
print(f"Validation images: {len(valid_paths)}; GNN fit: {len(fit_indices)}; threshold calibration: {len(calibration_indices)}")
print(f"Fit graphs with YOLO candidates: {len(fit_graphs)}")

Validation images: 600; GNN fit: 420; threshold calibration: 180
Fit graphs with YOLO candidates: 419


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class GraphSAGELayer(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.self_linear = nn.Linear(input_size, output_size)
        self.neighbor_linear = nn.Linear(input_size, output_size)

    def forward(self, x, edges):
        source, target = edges
        aggregate = torch.zeros_like(x)
        degree = torch.zeros((len(x), 1), dtype=x.dtype, device=x.device)
        if source.numel():
            aggregate.index_add_(0, target, x[source])
            degree.index_add_(0, target, torch.ones((len(target), 1), dtype=x.dtype, device=x.device))
        aggregate = aggregate / degree.clamp_min(1.0)
        return F.relu(self.self_linear(x) + self.neighbor_linear(aggregate))

class DetectionGraphScorer(nn.Module):
    def __init__(self, input_size=5 + NUM_CLASSES, hidden_size=32):
        super().__init__()
        self.layer1 = GraphSAGELayer(input_size, hidden_size)
        self.layer2 = GraphSAGELayer(hidden_size, hidden_size)
        self.dropout = nn.Dropout(0.15)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x, edges):
        x = self.dropout(self.layer1(x, edges))
        x = self.dropout(self.layer2(x, edges))
        return self.output(x).squeeze(-1)

def collate_graphs(graphs):
    features, labels, edge_parts = [], [], []
    offset = 0
    for graph in graphs:
        count = len(graph["x"])
        if count == 0:
            continue
        features.append(graph["x"])
        labels.append(graph["y"])
        if graph["edges"].numel():
            edge_parts.append(graph["edges"] + offset)
        offset += count
    if not features:
        return None
    x = torch.cat(features)
    y = torch.cat(labels)
    edges = torch.cat(edge_parts, dim=1) if edge_parts else torch.empty((2, 0), dtype=torch.long)
    return x, y, edges

positive_count = sum(int(graph["y"].sum()) for graph in fit_graphs)
negative_count = sum(len(graph["y"]) - int(graph["y"].sum()) for graph in fit_graphs)
if positive_count == 0:
    raise RuntimeError("No positive YOLO candidates in the GNN fit split; cannot train scorer")
pos_weight = min(20.0, negative_count / max(positive_count, 1))
scorer = DetectionGraphScorer().to(DEVICE)
optimizer = torch.optim.AdamW(scorer.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
EPOCHS = 15
GRAPH_BATCH_SIZE = 16

for epoch in range(EPOCHS):
    scorer.train()
    order = np.random.permutation(len(fit_graphs))
    epoch_losses = []
    for start in range(0, len(order), GRAPH_BATCH_SIZE):
        batch = [fit_graphs[i] for i in order[start:start + GRAPH_BATCH_SIZE]]
        collated = collate_graphs(batch)
        if collated is None:
            continue
        x, y, edges = (value.to(DEVICE) for value in collated)
        optimizer.zero_grad(set_to_none=True)
        logits = scorer(x, edges)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(scorer.parameters(), 5.0)
        optimizer.step()
        epoch_losses.append(float(loss.detach().cpu()))
    print(f"Epoch {epoch + 1:02d}/{EPOCHS}: loss={np.mean(epoch_losses):.4f}")

def score_graphs(graphs):
    scorer.eval()
    outputs = []
    with torch.no_grad():
        for graph in graphs:
            if len(graph["x"]) == 0:
                outputs.append(np.empty(0, dtype=np.float32))
                continue
            logits = scorer(graph["x"].to(DEVICE), graph["edges"].to(DEVICE))
            outputs.append(torch.sigmoid(logits).cpu().numpy())
    return outputs

def make_records(indices, detections, score_by_image):
    records = []
    for image_index in indices:
        det = detections[image_index]
        scores = score_by_image[image_index]
        if len(scores) != len(det["boxes"]):
            raise ValueError(f"Score count does not match detections for image index {image_index}")
        records.extend(
            (image_index, int(class_id), float(score), box)
            for box, class_id, score in zip(det["boxes"], det["classes"], scores)
        )
    return records

def match_records(records, ground_truth, iou_threshold=0.5):
    ordered = sorted(records, key=lambda row: row[2], reverse=True)
    true_positive = np.zeros(len(ordered), dtype=np.float32)
    matched = {}
    gt_lookup = []
    for objects in ground_truth:
        gt_lookup.append({c: [(i, box) for i, (gt_class, box) in enumerate(objects) if gt_class == c] for c in range(NUM_CLASSES)})
    for row_index, (image_index, class_id, _, box) in enumerate(ordered):
        key = (image_index, class_id)
        used = matched.setdefault(key, set())
        candidates = [(i, gt_box) for i, gt_box in gt_lookup[image_index][class_id] if i not in used]
        if not candidates:
            continue
        ids = [i for i, _ in candidates]
        gt_boxes = np.stack([gt_box for _, gt_box in candidates])
        ious = box_iou_one_to_many(box, gt_boxes)
        best = int(np.argmax(ious))
        if ious[best] >= iou_threshold:
            true_positive[row_index] = 1.0
            used.add(ids[best])
    false_positive = 1.0 - true_positive
    return ordered, true_positive, false_positive

def choose_f1_threshold(records, ground_truth, image_indices):
    ordered, tp, fp = match_records(records, ground_truth)
    if not ordered:
        raise RuntimeError("No detections available for threshold calibration")
    cumulative_tp = np.cumsum(tp)
    cumulative_fp = np.cumsum(fp)
    total_gt = sum(len(ground_truth[i]) for i in image_indices)
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1e-12)
    recall = cumulative_tp / max(total_gt, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    scores = np.asarray([row[2] for row in ordered])
    last_of_tie = np.r_[np.flatnonzero(scores[:-1] != scores[1:]), len(scores) - 1]
    best_position = last_of_tie[np.argmax(f1[last_of_tie])]
    return float(scores[best_position]), float(f1[best_position])

calibration_graphs = [valid_graphs[i] for i in calibration_indices]
calibration_gnn_scores_list = score_graphs(calibration_graphs)
calibration_gnn_scores = {i: s for i, s in zip(calibration_indices, calibration_gnn_scores_list)}
baseline_scores = {i: valid_detections[i]["scores"] for i in calibration_indices}
baseline_calibration_records = make_records(calibration_indices, valid_detections, baseline_scores)
gnn_calibration_records = make_records(calibration_indices, valid_detections, calibration_gnn_scores)
baseline_threshold, baseline_calibration_f1 = choose_f1_threshold(baseline_calibration_records, valid_ground_truth, calibration_indices)
gnn_threshold, gnn_calibration_f1 = choose_f1_threshold(gnn_calibration_records, valid_ground_truth, calibration_indices)
print(f"Frozen baseline threshold: {baseline_threshold:.4f} (calibration F1={baseline_calibration_f1:.3f})")
print(f"Frozen GNN threshold: {gnn_threshold:.4f} (calibration F1={gnn_calibration_f1:.3f})")

Epoch 01/15: loss=1.2541
Epoch 02/15: loss=1.1599
Epoch 03/15: loss=1.1355
Epoch 04/15: loss=1.0732
Epoch 05/15: loss=1.0106
Epoch 06/15: loss=0.9359
Epoch 07/15: loss=0.8260
Epoch 08/15: loss=0.7475
Epoch 09/15: loss=0.6991
Epoch 10/15: loss=0.6939
Epoch 11/15: loss=0.6469
Epoch 12/15: loss=0.6139
Epoch 13/15: loss=0.5825
Epoch 14/15: loss=0.5907
Epoch 15/15: loss=0.5902
Frozen baseline threshold: 0.1889 (calibration F1=0.535)
Frozen GNN threshold: 0.6753 (calibration F1=0.531)


In [4]:
test_dir, test_label_dir = SPLITS["test"]
test_paths = list_images(test_dir)
test_detections = predict_images(test_paths)
test_ground_truth = [read_ground_truth(p, test_label_dir) for p in test_paths]
test_graphs = graphs_for_paths(test_paths, test_detections)
test_gnn_scores_list = score_graphs(test_graphs)
test_indices = list(range(len(test_paths)))
test_baseline_scores = {i: test_detections[i]["scores"] for i in test_indices}
test_gnn_scores = {i: s for i, s in zip(test_indices, test_gnn_scores_list)}
test_baseline_records = make_records(test_indices, test_detections, test_baseline_scores)
test_gnn_records = make_records(test_indices, test_detections, test_gnn_scores)

def average_precision_from_matches(tp, fp, target_count):
    if target_count == 0:
        return 0.0
    cumulative_tp = np.cumsum(tp)
    cumulative_fp = np.cumsum(fp)
    recall = cumulative_tp / target_count
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1e-12)
    return float(np.mean([
        np.max(precision[recall >= threshold]) if np.any(recall >= threshold) else 0.0
        for threshold in np.linspace(0.0, 1.0, 101)
    ]))

def evaluate_method(records, ground_truth, score_threshold):
    selected = [row for row in records if row[2] >= score_threshold]
    ordered, tp, fp = match_records(selected, ground_truth, iou_threshold=0.5)
    per_class = {}
    iou_thresholds = np.arange(0.50, 0.96, 0.05)
    for class_id, class_name in enumerate(CLASS_NAMES):
        class_targets = sum(sum(gt_class == class_id for gt_class, _ in objects) for objects in ground_truth)
        class_tp = sum(int(flag) for row, flag in zip(ordered, tp) if row[1] == class_id)
        class_fp = sum(int(flag) for row, flag in zip(ordered, fp) if row[1] == class_id)
        class_fn = class_targets - class_tp
        precision = class_tp / max(class_tp + class_fp, 1)
        recall = class_tp / max(class_targets, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        ap_values = []
        class_records = [row for row in records if row[1] == class_id]
        for iou_threshold in iou_thresholds:
            ap_ordered, ap_tp, ap_fp = match_records(class_records, ground_truth, float(iou_threshold))
            ap_values.append(average_precision_from_matches(ap_tp, ap_fp, class_targets))
        per_class[class_name] = {
            "precision": precision, "recall": recall, "f1": f1,
            "tp": class_tp, "fp": class_fp, "fn": class_fn,
            "map50": ap_values[0], "map50_95": float(np.mean(ap_values)),
        }
    total_tp = sum(m["tp"] for m in per_class.values())
    total_fp = sum(m["fp"] for m in per_class.values())
    total_fn = sum(m["fn"] for m in per_class.values())
    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "precision": precision, "recall": recall, "f1": f1,
        "tp": total_tp, "fp": total_fp, "fn": total_fn,
        "map50": float(np.mean([m["map50"] for m in per_class.values()])),
        "map50_95": float(np.mean([m["map50_95"] for m in per_class.values()])),
        "classes": per_class,
    }

baseline_metrics = evaluate_method(test_baseline_records, test_ground_truth, baseline_threshold)
gnn_metrics = evaluate_method(test_gnn_records, test_ground_truth, gnn_threshold)
print(f"Test images: {len(test_paths)}; test labels were not used for training or threshold selection.")
print(f"{'Method':<19} {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>8} {'mAP50':>9} {'mAP50-95':>10} {'FP':>7} {'FN':>7}")
for name, metrics, threshold in [
    ("YOLO baseline", baseline_metrics, baseline_threshold),
    ("YOLO + GNN", gnn_metrics, gnn_threshold),
]:
    print(f"{name:<19} {threshold:>10.4f} {metrics['precision']:>10.3f} {metrics['recall']:>10.3f} {metrics['f1']:>8.3f} {metrics['map50']:>9.3f} {metrics['map50_95']:>10.3f} {metrics['fp']:>7d} {metrics['fn']:>7d}")
    for class_name, class_metrics in metrics["classes"].items():
        print(f"  class {class_name}: P={class_metrics['precision']:.3f}, R={class_metrics['recall']:.3f}, F1={class_metrics['f1']:.3f}, TP={class_metrics['tp']}, FP={class_metrics['fp']}, FN={class_metrics['fn']}, mAP50={class_metrics['map50']:.3f}, mAP50-95={class_metrics['map50_95']:.3f}")

Test images: 300; test labels were not used for training or threshold selection.
Method               Threshold  Precision     Recall       F1     mAP50   mAP50-95      FP      FN
YOLO baseline           0.1889      0.621      0.565    0.592     0.459      0.153    1796    2266
  class 3: P=0.541, R=0.573, F1=0.557, TP=251, FP=213, FN=187, mAP50=0.490, mAP50-95=0.184
  class 5: P=0.638, R=0.605, F1=0.621, TP=2502, FP=1421, FN=1633, mAP50=0.581, mAP50-95=0.195
  class 6: P=0.544, R=0.302, F1=0.388, TP=193, FP=162, FN=446, mAP50=0.306, mAP50-95=0.081
YOLO + GNN              0.6753      0.621      0.559    0.588     0.445      0.147    1774    2301
  class 3: P=0.598, R=0.493, F1=0.541, TP=216, FP=145, FN=222, mAP50=0.475, mAP50-95=0.178
  class 5: P=0.631, R=0.608, F1=0.619, TP=2514, FP=1472, FN=1621, mAP50=0.567, mAP50-95=0.185
  class 6: P=0.536, R=0.283, F1=0.371, TP=181, FP=157, FN=458, mAP50=0.294, mAP50-95=0.078
